In [1]:
import os

In [5]:
%pwd

'c:\\Users\\deepp\\end-to-end-ml-project\\research'

In [6]:
os.chdir("../")

In [7]:
%pwd

'c:\\Users\\deepp\\end-to-end-ml-project'

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL: str
    local_data_file:Path
    unzip_dir:Path

In [18]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories


In [10]:
from box import ConfigBox


In [17]:
import sys
from pathlib import Path

# Find project root by folder name
current = Path.cwd()
while current.name != "end-to-end-ml-project":
    if current.parent == current:
        raise RuntimeError("Project root not found")
    current = current.parent

PROJECT_ROOT = current
SRC_PATH = PROJECT_ROOT / "src"

print("Project root:", PROJECT_ROOT)
print("Src path:", SRC_PATH)

sys.path.insert(0, str(SRC_PATH))




Project root: c:\Users\deepp\end-to-end-ml-project
Src path: c:\Users\deepp\end-to-end-ml-project\src


In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath= CONFIG_FILE_PATH,
        params_filepath= PARAMS_FILE_PATH,
        schema_filepath= SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

    
        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

        


In [42]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [43]:
class DataIngestion:
    def __init__(self,config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size:{get_size(Path(self.config.local_data_file))}")
            
            
    def extract_zip_file(self):
        """_summary_
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [44]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config= data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-12-21 17:53:56,527:INFO: common : yaml file:config\config.yaml loaded successfully]
[2025-12-21 17:53:56,536:INFO: common : yaml file:params.yaml loaded successfully]
[2025-12-21 17:53:56,540:INFO: common : yaml file:schema.yaml loaded successfully]
[2025-12-21 17:53:56,543:INFO: common : created directory at:artifacts]
[2025-12-21 17:53:56,549:INFO: common : created directory at:artifacts/data_ingestion]
[2025-12-21 17:54:06,371:INFO: 3327477602 : artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: BA76:1C99DF:2F9C0D:6565DE:6947E6E4
Accept-Ranges: bytes
Date: Sun, 21 Dec 